In [ ]:
import pandas as pd
import numpy as np
import random

import sys, os
from tqdm import tqdm

sys.path.append(os.path.join(os.getcwd(), 'sql_engine_connection'))

from engine_connect import get_engine

import seaborn as sns
import matplotlib.pyplot as plt

from datetime import datetime, timedelta

In [ ]:
data = pd.read_json('data/events')
data['collector_tstamp'] = data.collector_tstamp.apply(lambda x: datetime.utcfromtimestamp(x / 1000))

In [ ]:
data.head()

,partner_key,collector_tstamp,domain_userid,ab_slot1_variant,event_name,product_domain,product_id,variant_size_label,fitpredictor_fuxp_seed_brand,fitpredictor_fuxp_seed_category,fitpredictor_fuxp_seed_size,prediction_type,prediction_size,dvce_screenwidth
0,Partner B,2017-10-21 04:23:59,7fef686a7f46988a53fc280cc713d011f762482f,None,viewed_product,Female Bottoms,T63WE,None,None,None,None,None,None,320
1,Partner A,2018-04-14 22:22:59,fd4a2cb28ba50022d6a04a36ce8c9cc9ad932eb4,Test,viewed_product,Dresses,39053,None,None,None,None,mn,XS,1280
2,Partner A,2018-04-14 19:31:08,c4f893cba554ed80df6a92b7fe004b32558fd652,Control,viewed_product,Dresses,46595,None,None,None,None,None,None,375
3,Partner A,2018-04-13 17:07:26,02f2b1ac4f5afc7a96f9fa188f00ca8bee7b2510,Test,viewed_product,Dresses,3229646,None,None,None,None,None,None,1920
4,Partner B,2017-10-16 20:25:07,4b2098df581a0378bc599dd238fe1fe648685883,None,viewed_product,Male Tops,B2AEX,None,None,None,None,None,None,375


In [ ]:
data.event_name.unique()

array(['viewed_product', 'added_variant_to_cart', 'opened_editor',
       'ordered_variant', 'selected_size', 'completed_profiling',
       'opened_brand_list', 'selected_category', 'selected_brand'],
      dtype=object)

In [ ]:
df_A = data[data.partner_key == 'Partner A']
df_B = data[data.partner_key == 'Partner B']

In [ ]:
df_A_test = df_A[df_A.ab_slot1_variant == 'Test']
df_A_control = df_A[df_A.ab_slot1_variant == 'Control']

In [ ]:
# calculating 95% confidence interval [x, y] for CRU = CR_test - CR_control (CRU- conversion rate uplift)

# number of users who made an order
NU_test = df_A_test[df_A_test.event_name == 'ordered_variant'].domain_userid.nunique()
# sample size
S_test = df_A_test.domain_userid.nunique()

NU_control = df_A_control[df_A_control.event_name == 'ordered_variant'].domain_userid.nunique()
S_control = df_A_control.domain_userid.nunique()

CR_test = NU_test / S_test
CR_control = NU_control / S_control

CRU = CR_test - CR_control

In [ ]:
CRU

0.0008672487982021373

In [ ]:
# 1.96 is a z-score for 95% confidence level
x = CRU - 1.96 * np.sqrt((CR_test * (1 - CR_test)) / S_test + (CR_control * (1 - CR_control)) / S_control)
y = CRU + 1.96 * np.sqrt((CR_test * (1 - CR_test)) / S_test + (CR_control * (1 - CR_control)) / S_control)

In [ ]:
print(f'confidence interval for CRU: [{x}, {y}]')

confidence interval for CRU: [-0.0002501755446367934, 0.001984673141041068]
